## Welcome to Week 6 Day 3: Context Engineering with MCP

People used to talk about Prompt Engineering, but now it's given way to a new Skill "Context Engineering".

Philipp Schmid of Google DeepMind wrote the seminal post about Context Engineering:

https://www.philschmid.de/context-engineering

For this week, we will put Context Engineering into practice, heavily using MCP servers.

1. Long-term memory: a knowledge graph the agent writes to and reads back
2. Web search: fresh information from the live web
3. Agentic RAG: a vector store the agent fills from its own research, then searches
4. Integrations: connecting to a live external service, with a local fallback

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv(override=True)

True

### A quick note for Windows

Launching a local MCP server from a notebook hits one rough edge on Windows. The server writes its startup output to stderr, but inside a Windows Jupyter kernel that stream has no real file handle behind it, so the launch fails with an `io.UnsupportedOperation: fileno` error, while Mac and Linux are unaffected.

The fix is to send the server's stderr to the null device, so it always has somewhere real to write. We do that once in the next cell, which lets every cell below use `MCPServerStdio` exactly as the OpenAI Agents SDK documents it. On Mac and Linux it costs nothing beyond keeping the server's startup banner out of the notebook.

In [2]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

## Part 1: Long-term memory

Our first context source is memory. This is the official knowledge graph server: it stores entities, observations about them, and the relationships between them, and keeps them on disk between runs. We point it at `memory/memory.json`, so you can open that file and read the graph it builds.

That gives an agent something it normally lacks: a memory that outlives the conversation. The agent writes facts as it learns them and reads them back later.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory

In [ ]:
MemoryPath = os.path.abspath("memory/memory.json")
MemoryParams = {"command": "npx", "args": ["-y","@modelcontextprotocol/server-memory"], "env": {"MEMORY_FILE_PATH":MemoryPath}}

async with MCPServerStdio(params=MemoryParams, client_session_timeout_seconds=60) as Server:
    MemoryTools = Server.list_tools()

MemoryTools

<coroutine object _MCPServerWithClientSession.list_tools at 0x00000162C6B17760>

In [5]:
Instructions = "You use your entity tools as a persistent memory to store and recall information about conversations"
Request = """
          My name is Siddharth Begwani. I am an electrical engineer and an MBA. I live in the city of Kolkata, India.
          I have ~23 years of experience in the IT industry, mainly in the field of CRM. Most of my career has been
          spent at Cognizant Technology Solutions. I enjoy working with AI, particularly agentic AI.
          """
Model = "gpt-5.4-mini"

In [6]:
async with MCPServerStdio(params=MemoryParams, client_session_timeout_seconds=60) as Server:
    agent = Agent(name="MemoryAgent", instructions=Instructions, model=Model, mcp_servers=[Server])
    with trace("Conversation"):
        Result = await Runner.run(agent,Request)
    display(Markdown(Result.final_output))

Got it, Siddharth — I’ve stored that information for future conversations.

In [7]:
async with MCPServerStdio(params=MemoryParams, client_session_timeout_seconds=60) as Server:
    agent = Agent(name="MemoryAgent", instructions=Instructions, model=Model, mcp_servers=[Server])
    with trace("Conversation"):
        Result = await Runner.run(agent,"My name is Siddharth Begwani. What do you know about me?")
    display(Markdown(Result.final_output))

Here’s what I know about you, Siddharth Begwani:

- You’re an electrical engineer and MBA.
- You live in Kolkata, India.
- You have about 23 years of experience in the IT industry.
- Your main professional focus is CRM.
- You’ve spent most of your career at Cognizant Technology Solutions.
- You enjoy working with AI, especially agentic AI.

If you want, I can also help keep track of additional details about your background, interests, projects, or goals.

### Check out the trace

https://platform.openai.com/traces

## Part 2: Web search

A model only knows what it was trained on. Web search is the context source that keeps it current.

We use Tavily, a search API built for agents: it returns clean, ranked results an LLM can use directly. Tavily maintains its own MCP server, so connecting is a one-liner.

This needs a free API key:

1. Sign up at https://www.tavily.com
2. The free tier gives you 1,000 searches a month, with no credit card
3. Copy your API key (it starts with `tvly-`) and add it to your `.env` file:

`TAVILY_API_KEY=tvly-xxxx`

In [8]:
TavilyParams = {"command":"npx", "args": ["-y","tavily-mcp@latest"], "env":{"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}}

async with MCPServerStdio(params=TavilyParams, client_session_timeout_seconds=60) as Server:
    TavilyTools = await Server.list_tools()

TavilyTools

[Tool(name='tavily_search', title=None, description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month', 'year']}, 'start_date':

Tavily's server offers several tools (search, extract, crawl, map, research). For this lab we only want plain web search, so we restrict the server to `tavily_search`. The OpenAI Agents SDK lets you hand an agent just the tools you choose with a static tool filter. Curating an agent's tools like this is itself context engineering.

In [9]:
Instructions = "You search the web for information and briefly summarize the takeaways"
Request = f"Please research the latest news on ITC Ltd. stock price and summarize its outlook. \
    For context, the current date is {datetime.now().strftime("%Y-%m-%d")}"
Model = "gpt-5.4-mini"
SearchOnly = create_static_tool_filter(allowed_tool_names=["tavily_search"])

In [10]:
async with MCPServerStdio(params=TavilyParams, client_session_timeout_seconds=60, tool_filter=SearchOnly) as Server:
    agent = Agent(name="agent", instructions=Instructions, model=Model, mcp_servers=[Server])
    Result = await Runner.run(agent, Request)
    display(Markdown(Result.final_output))

Here’s a quick research summary on **ITC Ltd. (NSE: ITC)** and its latest stock outlook as of **2026-09-19**:

### Latest price / recent performance
- Recent quoted levels in the search results were around **₹262–₹260**.
- The stock has been trading in a fairly tight range recently, with nearby support around **₹256–₹253** and resistance around **₹265–₹269**.
- Technical commentary from EquityPandit suggested:
  - **Immediate support:** ~₹256.4
  - **Immediate resistance:** ~₹264.6
  - A break below support could weaken the trend; a close above resistance could trigger a breakout.

### Latest news / catalysts
- **Q1 FY27 results were weak on profit**, with net profit reportedly **down 27% YoY** due to higher taxation on cigarettes and weaker agri-business exports.
- At the same time, **FMCG revenue/profit remained resilient**, helping offset some pressure.
- A later market note said the stock **rose ~2% after results** because revenue growth was strong and brokers like **Nomura upgraded ITC to Buy**, citing attractive risk-reward and possible recovery in cigarette margins.
- Near-term risks flagged in coverage included:
  - imported inflation
  - weak monsoon / lower kharif sowing
  - West Asia tensions impacting agri exports

### Outlook
**Short term:** Neutral to mildly positive, but range-bound.  
- The stock appears to be consolidating near support, so traders may watch whether it can reclaim the **₹265+** zone.
- If support near **₹256** breaks, downside risk rises.

**Fundamental view:** Mixed but stable.  
- Positives: strong FMCG franchise, steady consumption demand, high dividend yield, strong return ratios.
- Negatives: cigarettes remain exposed to taxation/regulatory pressure, and agri exports have near-term macro risk.

**Overall takeaway:**  
ITC looks like a **defensive, income-oriented stock with limited near-term upside unless earnings momentum improves**, but it still has support from strong cash generation, dividends, and a diversified business mix.

If you want, I can also give you:
1. a **bull vs bear case** for ITC, or  
2. a **technical trading setup** with support/resistance and scenario levels.

## Part 3: Agentic RAG

RAG, retrieval augmented generation, means giving a model relevant documents to ground its answer. The usual setup loads documents into a vector store up front. Agentic RAG turns that around: the agent builds the knowledge base itself, deciding what is worth keeping and storing it as it works.

We use the official Qdrant MCP server. Qdrant is a vector database, and the server exposes two tools: one stores a piece of text, the other finds the most relevant stored text for a query. It runs fully locally here. `QDRANT_LOCAL_PATH` keeps everything on disk with no separate database to run, and it embeds text with a local model, so there is no extra API key. The first store or search downloads that small embedding model, so it pauses once on first use.

https://github.com/qdrant/mcp-server-qdrant

We hand one agent both Tavily and Qdrant: it researches a topic on the web, stores what it learns, then answers from its own knowledge base.

In [11]:
VectorDBPath = Path("memory/qdrant")
VectorStoreParams = {"command":"uvx", "args":["mcp-server-qdrant"],
                     "env":{"QDRANT_LOCAL_PATH":str(VectorDBPath), "COLLECTION_NAME":"knowledge"}}
async with MCPServerStdio(params=VectorStoreParams, client_session_timeout_seconds=120) as Server:
    VectorStoreTools = await Server.list_tools()

VectorStoreTools

[Tool(name='qdrant-find', title=None, description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user', inputSchema={'properties': {'query': {'description': 'What to search for', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='qdrant-store', title=None, description='Keep the memory for later use, when you are asked to remember something.', inputSchema={'properties': {'information': {'description': 'Text to store', 'title': 'Information', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'default': None, 'description': 'Extra metadata stored along with memorised information. Any json is accepted.', 'title': 'Metadata'}}, 'required': ['information'], 'type': 'object'}, outpu

In [12]:
INSTRUCTIONS = """
               You research topics on the web and build up a knowledge base for later. When you learn something
               worth keeping, store it in your knowledge base. When you are asked what you know, search your
               knowledge base and answer from it.
               """
Model = "gpt-5.4-mini"

async with MCPServerStdio(params=TavilyParams, client_session_timeout_seconds=60, tool_filter=SearchOnly) as SearchServer:
    async with MCPServerStdio(params=VectorStoreParams, client_session_timeout_seconds=120) as VectorServer:
        agent = Agent(name="Researcher", instructions=Instructions, model=Model, mcp_servers=[SearchServer,VectorServer])
        with trace("Research and Store"):
            Result = await Runner.run(agent,"Research the latest news on Hero Motocorp and store the key facts in your knowledge base",
                                     max_turns=20)
        display(Markdown(Result.final_output))

I researched the latest Hero MotoCorp news and stored the key facts in my knowledge base.

Key takeaways:
- August 2026 sales were 568,398 units, up 2.65% YoY, but below estimates.
- Scooter sales grew strongly, and exports were 26,093 units.
- Hero MotoCorp announced an additional investment of up to ₹1,758 crore in Ather Energy.
- It also expanded its global presence with a distribution partnership in El Salvador.
- Q1 FY27 showed strong growth: revenue up 36% YoY to ₹12,999 crore and PAT at ₹1,454 crore.
- EV production capacity was doubled to 30,000 units per month.

If you want, I can also turn this into a concise one-page briefing or a timeline of recent events.

The agent below has only the knowledge base, no web search. Whatever it tells us about Nvidia, it is recalling from what the first agent stored. That is the retrieval half of RAG.

In [13]:
async with MCPServerStdio(params=VectorStoreParams, client_session_timeout_seconds=120) as VectorServer:
    agent = Agent(name="Researcher", instructions=Instructions, model=Model, mcp_servers=[VectorServer])
    with trace("Retrieve"):
        Result = await Runner.run(agent, "Based on your knowledge base, what is the latest on Hero Motocorp")
    display(Markdown(Result.final_output))

As of my latest knowledge, Hero MotoCorp is still the largest two-wheeler manufacturer in India and remains focused on three main areas:

- **EV push:** It continues building out its electric vehicle presence through the **Vida** brand, with a gradual scale-up rather than an all-at-once launch.
- **Premium expansion:** Hero has been trying to move beyond its commuter-bike core into more premium motorcycles and scooters, including partnerships and new product launches.
- **International and diversification efforts:** It has been working on exports, new markets, and adjacent businesses to reduce dependence on the core Indian commuter segment.

A few broad trends to watch:
- **Competition:** TVS, Bajaj, and Royal Enfield remain strong rivals, especially in premium and performance segments.
- **EV adoption pace:** Hero’s success in electric scooters will depend on pricing, range, charging ecosystem, and execution.
- **Market shift:** The commuter motorcycle market is mature, so growth likely depends on premium products, exports, and EVs.

If you want, I can also give you:
1. a **recent financial/performance snapshot**,  
2. **latest product launches**, or  
3. a **competitive comparison with TVS/Bajaj**.

### Check out the trace

https://platform.openai.com/traces

## Part 4: Integrations

The last context source is a live external service. Here that is market data from Massive (formerly Polygon.io), a popular financial data provider that publishes its own MCP server.

Setting this up is optional. Massive offers a free API key with no credit card, giving you their real end of day market data:

1. Sign up at https://www.massive.com
2. Create an API key
3. Add it to your `.env` file:

`MASSIVE_API_KEY=xxxx`

If you would rather not sign up, the next cell falls back to a local market server, the same kind of MCP server we built on Day 2, which serves simulated prices so the rest of the lab still works.

In [23]:
load_dotenv(override=True)

True

In [26]:
import os
from agents.mcp import MCPServerStdio, MCPServerStreamableHttp

TapetideAPIKey = os.getenv("TAPETIDE_TOKEN")

if TapetideAPIKey:
    MarketServer = MCPServerStreamableHttp(
        name="Tapetide",
        params = {
            "url":"https://mcp.tapetide.com/mcp", 
            "timeout":60,
            "headers": {
                "Authorization":f"Bearer {TapetideAPIKey}"
            }
        },
        client_session_timeout_seconds=60
    )
else:
    MarketServer = MCPServerStdio(
        params = {"command": "uv", "args": ["run", "-m", "backend.market_server"]},
        client_session_timeout_seconds=60 
    )


async with MarketServer as Server:
    MarketTools = await Server.list_tools()

MarketTools

[Tool(name='search_stocks', title='Search Indian Stocks', description="Search ~8,200 Indian stocks by name, symbol, BSE code, or ISIN. Also matches common/brand names and post-rename aliases (e.g. 'Zomato'→ETERNAL, 'Infosys'→INFY). Results ranked by relevance (exact symbol > prefix > contains). Use this FIRST to find a stock's symbol before calling other tools.\n\nAFTER using this tool: offer to get a detailed profile (get_company_profile), live quote (get_stock_quote), or add to watchlist for any stock the user is interested in.", inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'REQUIRED. The search text — a company name, symbol, BSE code, or ISIN. Example: "reliance" or "TCS".'}, 'sector': {'type': 'string', 'description': "Filter by sector (e.g. 'Information Technology')"}, 'industry': {'type': 'string', 'description': "Filter by industry (e.g. 'Private Bank')"}, 'limit': {'type': 'number', 'default': 10, 'description': 'Max results (default 

In [ ]:
# massive_api_key = os.getenv("MASSIVE_API_KEY")

# if massive_api_key:
#     market_params = {
#         "command": "uvx",
#         # mcp_massive still imports mcp.server.fastmcp, an API the mcp SDK's 2.0.0 release removed.
#         # mcp_massive doesn't cap its mcp dependency, so pin uvx to an older, compatible mcp here.
#         "args": ["--with", "mcp<2.0.0", "--from", "git+https://github.com/massive-com/mcp_massive@v0.10.0", "mcp_massive"],
#         "env": {"MASSIVE_API_KEY": massive_api_key},
#     }
# else:
#     market_params = {"command": "uv", "args": ["run", "-m", "backend.market_server"]}

In [29]:
Instructions = "You answer questions about the stock market"
Request = "What are the support and resistance levels for Hero Motocorp? What was the price it was last traded at, and \
        what is its max pain point?"
Model = "gpt-5.4-mini"

MarketParams = {
    "url":"https://mcp.tapetide.com/mcp",
    "timeout":60,
    "headers": {
        "Authorization":f"Bearer {TapetideAPIKey}"
    }
}

async with MCPServerStreamableHttp(name="Tapetide", params=MarketParams, client_session_timeout_seconds=120) as TPTServer:
    agent = Agent(name="MarketData", instructions=Instructions, model=Model, mcp_servers=[TPTServer])
    with trace("Conversation"):
        Result = await Runner.run(agent,Request)
    display(Markdown(Result.final_output))

For **Hero MotoCorp (HEROMOTOCO)**:

- **Last traded price:** **₹5,285.5**
- **Support levels:** roughly **₹5,270**, **₹5,166**, then **₹5,000–₹5,030**
- **Resistance levels:** roughly **₹5,333–₹5,362**, then **₹5,400**, and higher up **₹5,550+**

**Max pain point:** **₹5,400**  
- For the nearest expiry (**2026-09-29**), max pain is **₹5,400**
- The next expiry (**2026-10-27**) also shows **₹5,400**

If you want, I can also give you:
- a cleaner **pivot-based support/resistance table**
- the **option chain around max pain**
- or a **technical trend view** for Hero MotoCorp.

## That's it for today

Four MCP servers, four kinds of context: long-term memory, web search, a knowledge base the agent builds itself, and a live integration. Choosing which sources an agent needs, and wiring them in, is much of what context engineering is in practice.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Browse an MCP marketplace like glama.ai/mcp or smithery.ai and add another context source to this notebook, using one of today's four patterns.
            </span>
        </td>
    </tr>
</table>